## Ingest sprints folder
1. Read the file using Dataframe reader API
    - Add Metadata Columns
    - Source File
3. Ingestion Timestamp
4. Write to bronze delta _table_

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
# source and table name 

source_file = f"{landing_folder_path}/sprints"
table_name = f"{catalog_name}.{bronze_schema}.sprints"

**Step 1 - Read the CSV file using the dataframe reader API**


In [0]:
#  define the schema type 

from pyspark.sql.types import StringType, StructType, StructField, DateType, IntegerType, FloatType



sprints_schema = StructType([
        StructField('date', DateType()),
        StructField('raceName', StringType()),
        StructField('round', IntegerType()),
        StructField('season', IntegerType()),
        StructField('url', StringType()),
        StructField('constructorId', StringType()),
        StructField('driverId', StringType()),
        StructField('grid', IntegerType()),
        StructField('laps', IntegerType()),
        StructField('number', IntegerType()),
        StructField('points', FloatType()),
        StructField('position', IntegerType()),
        StructField('positionText', StringType()),
        StructField('status', StringType())

    ])


In [0]:
sprints_df = ( spark.read
            .format('json')
            #.option('header', True)
            #.option('inferSchema', 'true')
            .option('mode', 'FAILFAST')
            .option('multiLine', True)
            .schema(sprints_schema)
            .load(source_file)
)





**Step 2 - Add the Metadata columns**
  - Source file
  - Ingestion Timestamp


In [0]:
sprints_df_final = add_ingestion_metadata(sprints_df)

**Step 3 - Write to bronze delta table**

In [0]:
(
    sprints_df_final
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(table_name)
)

In [0]:
display(spark.table(table_name))

In [0]:
%sql
SELECT season, COUNT(*)
    From Formula1.bronze.sprints 
GROUP BY season
ORDER BY season